In [0]:
%pylab inline

In [67]:
import dataikuapi
from typing import Dict, Any
import os
import io
import fitz  # PyMuPDF
import pdfplumber
from uuid import uuid4
from soa_extraction.opensearch_utils import OpensearchUtil

import sys
import os
import fitz  # PyMuPDF
# from langgraph_utils.Info_extractor import InfoExtractorAgent
# from langgraph_utils.chat_history import SnowflakeChatMessageHistory
import json
# from langgraph_utils.file_parser import FileParser
# from langgraph_utils.digitization import main_handler
import uuid
# from langgraph_utils import creds
# from langgraph_utils.variables import PROJECT_NAME, SECRET_NAME, TOKEN_KEY
from utils.connection import get_dataiku_client_and_project
import logging



DATAIKU_HOST = "http://10.45.152.66:10000"
API_SECRET_KEY = "dkuaps-b3EsRXVjU3w4y7nd4KEwibEr04CjFPZr"          
PROJECT_NAME = "ECSGENERATION"   

# client, proj = get_dataiku_client_and_project(PROJECT_NAME, SECRET_NAME, TOKEN_KEY)
client = dataikuapi.DSSClient(DATAIKU_HOST, API_SECRET_KEY)
proj = client.get_project(PROJECT_NAME)

In [23]:
import os
import io
import fitz  # PyMuPDF
import pdfplumber
from uuid import uuid4


class historical_CRF:
    
    def __init__(self, client, proj, chunk_size=1000):
        self.proj = proj
        self.client = client
        self.s3_folder_dataset_id = proj.get_variables()['local'].get('file_upload') # change file upload 
        self.input_folder = proj.get_managed_folder(self.s3_folder_dataset_id)
        self.files = self.input_folder.list_contents()["items"]
        self.toc_page_limit = 20
        self.config = proj.get_variables()["local"]
        print(self.files)

    def historical_mapping(self, file_path,paths=[]):
        result = []

#         for file in paths:
#             path = file
#             print(path)
#             parts = path.strip('/').split('/')

#             if "Historical" in path:
                
#                 result.append({
                    
#                     "path": path,
                   
    
#                 })
        result.append(file_path)

        response = []

        for i in result:
            

            with self.input_folder.get_file(file_path) as stream:
                file_bytes = stream.raw.data
            
            
            try:
                
                import re

                def clean_summary(text):
                    # Remove newlines, tabs, and collapse extra spaces
                    text = re.sub(r'\s+', ' ', text).strip()

                    # Ensure it ends with a single period
                    if not text.endswith('.'):
                        text += '.'

                    return text
                
                pdf_file_like = io.BytesIO(file_bytes)
                with pdfplumber.open(pdf_file_like) as pdf:
                    
                    for page_num, page in enumerate(pdf.pages):
                        res = {}
#                         parts = i["path"].strip('/').split('/')
                        therapeutic_area = ''
                        source =  "Unknown"
                        template_name = os.path.basename(file_path)
                        unique_id = uuid4()
#                         print(hist_id)
                        res = {
                           
                            
                            "template_name": template_name,
                            "path": file_path,
                            "id": unique_id,

                        }
                        
                        page_text = page.extract_text(layout=True)
                        
                        if page_text and "field name" in page_text.lower():
                            continue

                        if not page_text:
                            continue

                        lines = page_text.split('\n')
                        header_lines = []
                        field_value_map = {}
                        current_field = ""
                        in_field_section = False

                        for line in lines:
                            line = line.strip()
                            if not line:
                                continue

                            # Trigger point for header vs fields
                            if not in_field_section:
                                if "generated" in line.lower():
                                    in_field_section = True
                                    continue
                                header_lines.append(line)
                            else:
                                if len(line.strip()) == 0:
                                    continue
                                
                                pattern = r'''
                                    ^                              # Start of line
                                    (?P<field>.+?)                 # Field name (non-greedy)
                                    (?:\t|\s{2,})+                 # Separator: tab or ≥2 spaces
                                    (?P<value>.+?)                 # Value
                                    \s*$                           # Optional trailing spaces
                                '''
                                pattern2  = r'^(?!\s)(?!.*\s$)(?P<value>.+)$'

                                
                                
                                
                                
                                
#                                 current_field = None

                                # Step 1 ─ collect every “proper” field line
                                m = re.match(pattern, line, re.VERBOSE)
                                p = re.match(pattern2, line, re.VERBOSE) 
                                if m:
                                    if m.group("field") and m.group("value"):
                                        feild = m.group("field")
                                        current_field = feild
                                        value = m.group("value")
                                        
                                        
                                if p:
                                    if p.group("value") and current_field:
                                        # Continuation of previous field
#                                         feild = current_field
                                        value = p.group("value")
                                        
                        
                               
                                
                                left_part = current_field.strip()
                                right_part = value.strip()

                                if left_part:
                                    if left_part in field_value_map and right_part:
                                        field_value_map[left_part].append(right_part)
                                     
                                    else:
                                        
                                        field_value_map[left_part] = [right_part.strip()]
                             

                        final_feilds = []
                        for k in field_value_map:
                            
                            final_feilds.append({
                                "field_name" : k,
                                "field_value": field_value_map[k]
                            })
                        
                        head = ""
                        for j in header_lines:
                            if "form" in j.lower().strip() or "folder" in j.lower().strip():
                                head += j + " "
                        
                        res["source_data"] = {
                            "assessments" : head,
                            "feilds" : final_feilds 
                            
                        }
#                         print(res)
                    
#                     print(res)
                        if final_feilds and head:
                            response.append(res)
#                             print(f"✅ extraction for {i['path']} completed")
                    

            except Exception as e:
                print(f"Error processing file {i['path']}: {e}")
        
        

        return response


In [24]:
obj = historical_CRF(client , proj)
file_path = '/Annotated_Otsuka_405 201 00157_00150405_v1.0_Complete eCRF (1).pdf'

[{'path': '/Annotated_Otsuka_405 201 00157_00150405_v1.0_Complete eCRF (1).pdf', 'size': 2544020, 'lastModified': 1757496080000}]


In [42]:
import time
import re
start = time.time()
final_list = []
response = obj.historical_mapping(file_path)

for resp in response:
    form_name = resp['source_data']['assessments']
    match = re.search(r'Form[:\s]*(.*)', form_name)
    if match:
#         print(match.group(1))
        form_name = match.group(1)
    for field in resp['source_data']['feilds']:
        field_name = field['field_name']
        final_list.append({
            "form_name":form_name,
            "field_name":field_name
        })
end = time.time()
tt = end-start

In [32]:
response

[{'therapeutic_area': '',
  'source': 'Unknown',
  'template_name': 'Annotated_Otsuka_405 201 00157_00150405_v1.0_Complete eCRF (1).pdf',
  'path': '/Annotated_Otsuka_405 201 00157_00150405_v1.0_Complete eCRF (1).pdf',
  'id': UUID('5d55fbba-80c9-4130-a212-16718bb12953'),
  'protocol_summary': '',
  'historical_crf_id': '',
  'source_data': {'assessments': 'Form: Enrollment ',
   'feilds': [{'field_name': 'Site ID',
     'field_value': ['Site ID                                                               001  1',
      '002']},
    {'field_name': 'Participant ID',
     'field_value': ['Participant ID                                                             2']},
    {'field_name': 'Participant Number (Derived)',
     'field_value': ['Participant Number (Derived)                                               3',
      'Otsuka_405 201 00157_00150405_v1.0',
      '1 of 640',
      '(1410)']}]}},
 {'therapeutic_area': '',
  'source': 'Unknown',
  'template_name': 'Annotated_Otsuka_405

In [43]:
len(final_list)


3307

In [0]:
final_list

In [44]:
tt

37.82287645339966

In [45]:
import pandas as pd 
pd.set_option("display.max_rows",None)
data = pd.DataFrame(final_list)

In [46]:
data

,form_name,field_name
0,Enrollment,Site ID
1,Enrollment,Participant ID
2,Enrollment,Participant Number (Derived)
3,Date of Visit,Visit date
4,Informed Consent,Informed consent obtained?
5,Informed Consent,Informed consent date
6,Informed Consent,Informed consent time
7,Informed Consent,Derived date
8,Informed Consent,Informed consent version number
9,Informed Consent,Standardized disposition term


In [117]:
sub_data = data.iloc[:101]

In [80]:
opensearch_client = OpensearchUtil(client,proj)

{'type': 'ElasticSearch', 'params': {'host': 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com', 'username': 'genai-admin', 'password': 'qPk7Jf5vcyXDS!**332gXTvSfmcauvr9', 'port': 443, 'ssl': True, 'trustAnySSLCertificate': True, 'dialect': 'ES_7', 'dkuProperties': [], 'namingRule': {'indexNameDatasetNamePrefix': '${projectKey}_'}, 'authType': 'PASSWORD', 'oauth': {'refreshTokenRotation': False}, 'aws': {'service': 'OPENSEARCH_SERVERLESS', 'credentialsMode': 'KEYPAIR', 'customAWSCredentialsProviderParams': []}}, 'credentialsMode': 'GLOBAL', 'proxySettingsAsString': ''}
opensearch <OpenSearch([{'host': 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com', 'port': 443}])>


In [193]:
form = data.iloc[24]['form_name']
field = data.iloc[24]['field_name']
form_emb = opensearch_client.create_embedding(form,proj.get_variables()['local'].get("default_embeddings_model_id"))
field_emb = opensearch_client.create_embedding(field,proj.get_variables()['local'].get("default_embeddings_model_id"))

In [194]:
form_vec = form_emb['response']
field_vec = field_emb['response']
print(form,field)

Demographics  Combined (Estrogen- and Progestogen- containing) hormonal contraception


In [195]:
query = {
    "query": {
        "bool": {
            "should": [
                {
                    "knn": {
                        "form_name_vector": {
                            "vector": form_vec,
                            "k": 10
                        }
                    }
                },
                {
                    "knn": {
                        "form_field_value_vector": {
                            "vector": field_vec,
                            "k": 10
                        }
                    }
                }
            ]
        }
    }
}

In [196]:
index_name = proj.get_variables()['local'].get('ecs_opensearch')
dataiku_project_var = "${projectKey}"
if dataiku_project_var in index_name:
    index_name = index_name.replace(dataiku_project_var, opensearch_client.project.project_key).lower()

In [197]:
p = opensearch_client.opensearch_client

In [198]:
p = opensearch_client.opensearch_client
result = p.search(index=index_name,body=query,size=2)

/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(


In [199]:
output_list = []
for hit in result["hits"]["hits"]:
    print(hit["_id"], hit["_score"])
    output_list.append(hit['_source'])


087e42b4-ca61-417f-9abf-5ec9d533f277 1.5120363
36d5dcbe-3f0d-479e-8e42-5b8369fe09b5 1.5120363


In [200]:
dff = pd.DataFrame(output_list)
print(form,field)

Demographics  Combined (Estrogen- and Progestogen- containing) hormonal contraception


In [201]:
dff

,ecs_id,form_id,validation_id,form_name,form_domain_name,form_field_value,variable_name,validation_logic,reasoning,action,action_details,source,path,form_name_vector,form_field_value_vector
0,087e42b4-ca61-417f-9abf-5ec9d533f277,4a5552de-4909-4846-bd25-3c28f9351b53,MVAL_DM004,Demographics,DM,"If sex is female, is the subject of childbeari...",RPORRES_CHILDPOT,"(DM.SEX == ""Female"") then (DM.RPORRES_CHILDPOT...","If female then ""If sex is female, is the subje...",DM.RPORRES_CHILDPOT is enterable,<n/a>,Standard,/Standard/Copy of Otsuka Standard Edit Check S...,"[0.00561428, 0.014669912, 0.0441228, 0.056164,...","[0.07271021, -0.003000886, 0.0149580585, 0.070..."
1,36d5dcbe-3f0d-479e-8e42-5b8369fe09b5,4a5552de-4909-4846-bd25-3c28f9351b53,MVAL_DM022,Demographics,DM,"If sex is female, is the subject of childbeari...",RPORRES_CHILDPOT,(DM.RPORRES_CHILDPOT is enterable and missing),Field must not be missing when enterable,prompt user with ACTION DETAILS,<query the field for missing data>,Standard,/Standard/Copy of Otsuka Standard Edit Check S...,"[0.00561428, 0.014669912, 0.0441228, 0.056164,...","[0.07271021, -0.003000886, 0.0149580585, 0.070..."


In [202]:
client  = opensearch_client.opensearch_client
index_name = proj.get_variables()['local'].get('ecs_opensearch')
dataiku_project_var = "${projectKey}"
if dataiku_project_var in index_name:
    index_name = index_name.replace(dataiku_project_var, opensearch_client.project.project_key).lower()
    
msearch_body = []

for index , row in sub_data.iterrows():
    form_name = row['form_name']
    field_value = row['field_name']
    form_emb = opensearch_client.create_embedding(form_name,proj.get_variables()['local'].get("default_embeddings_model_id"))
    field_emb = opensearch_client.create_embedding(field_name,proj.get_variables()['local'].get("default_embeddings_model_id"))
    
    form_vec = form_emb['response']
    field_vec = field_emb['response']
    msearch_body.append({"index": index_name})
    msearch_body.append({
        "query": {
            "bool": {
                "should": [
                    {
                        "knn": {
                            "form_name_vector": {
                                "vector": form_vec,
                                "k": 5
                            }
                        }
                    },
                    {
                        "knn": {
                            "form_field_value_vector": {
                                "vector": field_vec,
                                "k": 5
                            }
                        }
                    }
                ]
            }
        },
        "size": 1
    })
    
    

In [203]:
response = client.msearch(body=msearch_body)
response_list = []
# Print results for each record
for i, res in enumerate(response["responses"]):
#     print(f"\n🔹 Results for record {i+1} ({records[i]['form_name']} / {records[i]['field_name']}):")
    for hit in res["hits"]["hits"]:
#         print(hit["_id"], hit["_score"])
#         print(sub_data.iloc[i])
        hit['_source']['original_form_name'] = sub_data.iloc[i]['form_name']
        hit['_source']['original_field_value'] = sub_data.iloc[i]['field_name']
        response_list.append(hit['_source'])

/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(


In [149]:
len(response_list)


101

In [204]:
res_df = pd.DataFrame(response_list)

In [160]:
s = res_df.drop_duplicates(subset=['form_name','form_field_value'])

In [205]:
print(res_df.columns)

res_df = res_df[["original_form_name", "original_field_value","form_name","form_field_value"]]
res_df

Index(['ecs_id', 'form_id', 'validation_id', 'form_name', 'form_domain_name',
       'form_field_value', 'variable_name', 'validation_logic', 'reasoning',
       'action', 'action_details', 'source', 'path', 'form_name_vector',
       'form_field_value_vector', 'original_form_name',
       'original_field_value'],
      dtype='object')


,original_form_name,original_field_value,form_name,form_field_value
0,Enrollment,Site ID,Adverse Event,Is the adverse event Life Threatening?
1,Enrollment,Participant ID,Adverse Event,Is the adverse event Life Threatening?
2,Enrollment,Participant Number (Derived),Adverse Event,Is the adverse event Life Threatening?
3,Date of Visit,Visit date,Subject Visits,Visit Time
4,Informed Consent,Informed consent obtained?,Informed Consent,Date of Consent
5,Informed Consent,Informed consent date,Informed Consent,Date of Consent
6,Informed Consent,Informed consent time,Informed Consent,Date of Consent
7,Informed Consent,Derived date,Informed Consent,Date of Consent
8,Informed Consent,Informed consent version number,Informed Consent,Date of Consent
9,Informed Consent,Standardized disposition term,Informed Consent,Date of Consent


In [144]:
sub_data

,form_name,field_name
0,Enrollment,Site ID
1,Enrollment,Participant ID
2,Enrollment,Participant Number (Derived)
3,Date of Visit,Visit date
4,Informed Consent,Informed consent obtained?
5,Informed Consent,Informed consent date
6,Informed Consent,Informed consent time
7,Informed Consent,Derived date
8,Informed Consent,Informed consent version number
9,Informed Consent,Standardized disposition term


In [219]:
#from fuzzyness
import time 
start = time.time()
client  = opensearch_client.opensearch_client
index_name = proj.get_variables()['local'].get('ecs_opensearch')
dataiku_project_var = "${projectKey}"
if dataiku_project_var in index_name:
    index_name = index_name.replace(dataiku_project_var, opensearch_client.project.project_key).lower()
    
msearch_body = []

for index , row in sub_data.iterrows():
    form_name = row['form_name']
    field_value = row['field_name']
    form_emb = opensearch_client.create_embedding(form_name,proj.get_variables()['local'].get("default_embeddings_model_id"))
    field_emb = opensearch_client.create_embedding(field_name,proj.get_variables()['local'].get("default_embeddings_model_id"))
    
    form_vec = form_emb['response']
    field_vec = field_emb['response']
    msearch_body.append({"index": index_name})
    msearch_body.append({
    "query": {
        "bool": {
            "must": [
                {
                    "knn": {
                        "form_name_vector": {
                            "vector": form_vec,
                            "k": 50        # broader pool of vector candidates
                        }
                    }
                },
                {
                    "knn": {
                        "form_field_value_vector": {
                            "vector": field_vec,
                            "k": 50
                        }
                    }
                }
            ],
            "should": [
                {
                    "match": {
                        "form_name": {
                            "query": form_name,
                            
                           
                        }
                    }
                },
                {
                    "match_phrase": {
                        "form_field_value": {
                            "query": field_value,
                          "slop":1
                            
                        }
                    }
                }
            ]
        }
    },
    "size": 1   # get top 5 after rescoring
})


response = client.msearch(body=msearch_body)
response_list = []
# Print results for each record
for i, res in enumerate(response["responses"]):
#     print(f"\n🔹 Results for record {i+1} ({records[i]['form_name']} / {records[i]['field_name']}):")
    for hit in res["hits"]["hits"]:
#         print(hit["_id"], hit["_score"])
#         print(sub_data.iloc[i])
        hit['_source']['original_form_name'] = sub_data.iloc[i]['form_name']
        hit['_source']['original_field_value'] = sub_data.iloc[i]['field_name']
        response_list.append(hit['_source'])    
end = time.time()
tt = end-start

/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(


In [220]:
print(res_df.columns)
res_df = pd.DataFrame(response_list)
res_df = res_df[["original_form_name", "original_field_value","form_name","form_field_value"]]
res_df

Index(['original_form_name', 'original_field', 'form_name',
       'form_field_value'],
      dtype='object')


,original_form_name,original_field_value,form_name,form_field_value
0,Enrollment,Site ID,Demographics,Birth Time (24 hr clock)
1,Enrollment,Participant ID,Demographics,Birth Time (24 hr clock)
2,Enrollment,Participant Number (Derived),Demographics,Birth Time (24 hr clock)
3,Date of Visit,Visit date,Subject Visits,Visit Date
4,Informed Consent,Informed consent obtained?,Informed Consent,Was informed consent obtained?
5,Informed Consent,Informed consent date,Informed Consent,Type of Consent
6,Informed Consent,Informed consent time,Informed Consent,Type of Consent
7,Informed Consent,Derived date,Informed Consent,Type of Consent
8,Informed Consent,Informed consent version number,Informed Consent,Type of Consent
9,Informed Consent,Standardized disposition term,Informed Consent,Type of Consent


In [221]:
tt

4.447622299194336

In [190]:
#new index 
#from fuzzyness
index_name = ""
import time 
start = time.time()
client  = opensearch_client.opensearch_client
# index_name = proj.get_variables()['local'].get('ecs_opensearch')
index_name = "${projectKey}_test_ecs"
dataiku_project_var = "${projectKey}"
if dataiku_project_var in index_name:
    index_name = index_name.replace(dataiku_project_var, opensearch_client.project.project_key).lower()
    
msearch_body = []

for index , row in sub_data.iterrows():
    form_name = row['form_name']
    field_value = row['field_name']
    mix = f"{form_name} : {field_value}"
    print(mix)
    mix_emb = opensearch_client.create_embedding(mix,proj.get_variables()['local'].get("default_embeddings_model_id"))
    #field_emb = opensearch_client.create_embedding(field_name,proj.get_variables()['local'].get("default_embeddings_model_id"))
    
    mix_vec = mix_emb['response']
#     field_vec = field_emb['response']
    msearch_body.append({"index": index_name})
    msearch_body.append({
    "query": {
        "bool": {
            "must": [
                {
                    "knn": {
                        "form_field_vector": {
                            "vector": form_vec,
                            "k": 5      # broader pool of vector candidates
                        }
                    }
                }
            ]
        }
    },
    "size": 1   # get top 1 result after rescoring
})


response = client.msearch(body=msearch_body)
response_list = []
# Print results for each record
for i, res in enumerate(response["responses"]):
#     print(f"\n🔹 Results for record {i+1} ({records[i]['form_name']} / {records[i]['field_name']}):")
    for hit in res["hits"]["hits"]:
#         print(hit["_id"], hit["_score"])
        print(sub_data.iloc[i])
        hit['_source']['original_form_name'] = sub_data.iloc[i]['form_name']
        hit['_source']['original_field_value'] = sub_data.iloc[i]['field_name']
        response_list.append(hit['_source'])    
end = time.time()
tt = end-start

Enrollment  : Site ID
Enrollment  : Participant ID
Enrollment  : Participant Number (Derived)
Date of Visit  : Visit date
Informed Consent  : Informed consent obtained?
Informed Consent  : Informed consent date
Informed Consent  : Informed consent time
Informed Consent  : Derived date
Informed Consent  : Informed consent version number
Informed Consent  : Standardized disposition term
Informed Consent  : Protocol version
Demographics  : Date of Birth
Demographics  : Age
Demographics  : Sex at Birth
Demographics  : Gender Identity
Demographics  : If Female, is the participant of childbearing potential?
Demographics  : If No, provide reason
Demographics  : If Other, Specify
Demographics  : If Male, is the participant of reproductive potential?
Demographics  : Intrauterine device (IUD)
Demographics  : Intrauterine hormone-releasing system (IUS)
Demographics  : Bilateral tubal occlusion/ligation
Demographics  : Azoospermia or Azoospermic partner
Demographics  : Sexual abstinence
Demographi

/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(


In [248]:
print(res_df.columns)
res_df = pd.DataFrame(response_list)
res_df = res_df[["original_form_name", "original_field_value","form_name","form_field_value"]]
res_df.iloc[31]['original_field_value']

Index(['original_form_name', 'original_field_value', 'form_name',
       'form_field_value'],
      dtype='object')


'Did participant satisfy all Inclusion/Exclusion criteria?'

In [244]:
# single search 
start = time.time()
client  = opensearch_client.opensearch_client
output_list = []
index_name = proj.get_variables()['local'].get('ecs_opensearch')
dataiku_project_var = "${projectKey}"
if dataiku_project_var in index_name:
    index_name = index_name.replace(dataiku_project_var, opensearch_client.project.project_key).lower()
for index , row in sub_data.iterrows():
    form_name = row['form_name']
    field_name = row['field_name']
    form_emb = opensearch_client.create_embedding(form_name,proj.get_variables()['local'].get("default_embeddings_model_id"))
    field_emb = opensearch_client.create_embedding(field_name,proj.get_variables()['local'].get("default_embeddings_model_id"))
    
    form_vec = form_emb['response']
    field_vec = field_emb['response']
    query = {
    "query": {
        "bool": {
            "should": [
                {
                    "knn": {
                        "form_name_vector": {
                            "vector": form_vec,
                            "k": 10
                        }
                    }
                },
                {
                    "knn": {
                        "form_field_value_vector": {
                            "vector": field_vec,
                            "k": 10
                        }
                    }
                }
            ]
        }
    }
}
    p = opensearch_client.opensearch_client
    result = p.search(index=index_name,body=query,size=1)
    for hit in result["hits"]["hits"]:
        print(hit["_id"], hit["_score"])
        hit['_source']['original_form_name'] = form_name
        hit['_source']['original_field'] = field_name
        print(hit['_source'])
        if hit['_score'] < 0.75:
            pass
            # make the llm call 
        output_list.append(hit['_source'])
    break
final_df = pd.DataFrame(output_list)
end = time.time()

44c40b95-e715-4892-bcd4-6400447a965e 1.1995912
{'ecs_id': '44c40b95-e715-4892-bcd4-6400447a965e', 'form_id': '4a5552de-4909-4846-bd25-3c28f9351b53', 'validation_id': 'MVAL_DM027', 'form_name': 'Demographics', 'form_domain_name': 'DM', 'form_field_value': 'Country', 'variable_name': 'COUNTRY', 'validation_logic': '(DM.COUNTRY is enterable and missing)', 'reasoning': 'Field must not be missing when enterable', 'action': 'prompt user with ACTION DETAILS', 'action_details': '<query the field for missing data>', 'source': 'Standard', 'path': '/Standard/Copy of Otsuka Standard Edit Check Specifications (1).xlsx', 'form_name_vector': [0.00561428, 0.014669912, 0.0441228, 0.056164, 0.05760555, -0.0077872076, -0.038328324, 0.020012135, -0.01243692, 0.019079365, 0.011871605, -0.0012516417, -0.039148033, -0.005448219, 0.02794067, -0.010507784, -0.013503951, 0.020308925, 0.014811241, 0.029113699, 0.04398147, 0.026838308, 0.028477719, -0.0089178365, -0.0076388125, 0.050821777, -0.056192264, -0.02081

/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(


In [216]:
print(end-start)
print(res_df.columns)
# res_df = pd.DataFrame(response_list)
res_df = final_df[["original_form_name", "original_field","form_name","form_field_value"]]
res_df

5.450715780258179
Index(['original_form_name', 'original_field', 'form_name',
       'form_field_value'],
      dtype='object')


,original_form_name,original_field,form_name,form_field_value
0,Enrollment,Site ID,Demographics,Country
1,Enrollment,Participant ID,Demographics,Age Unit
2,Enrollment,Participant Number (Derived),Demographics,Age Unit
3,Date of Visit,Visit date,Subject Visits,Visit Date
4,Informed Consent,Informed consent obtained?,Informed Consent,Was informed consent obtained?
5,Informed Consent,Informed consent date,Informed Consent,Was informed consent obtained?
6,Informed Consent,Informed consent time,Informed Consent,Was informed consent obtained?
7,Informed Consent,Derived date,Medical History,End Date
8,Informed Consent,Informed consent version number,Informed Consent,Was informed consent obtained?
9,Informed Consent,Standardized disposition term,Informed Consent,Type of Consent


In [213]:
final_df

,ecs_id,form_id,validation_id,form_name,form_domain_name,form_field_value,variable_name,validation_logic,reasoning,action,action_details,source,path,form_name_vector,form_field_value_vector,original_form_name,original_field
0,44c40b95-e715-4892-bcd4-6400447a965e,4a5552de-4909-4846-bd25-3c28f9351b53,MVAL_DM027,Demographics,DM,Country,COUNTRY,(DM.COUNTRY is enterable and missing),Field must not be missing when enterable,prompt user with ACTION DETAILS,<query the field for missing data>,Standard,/Standard/Copy of Otsuka Standard Edit Check S...,"[0.00561428, 0.014669912, 0.0441228, 0.056164,...","[-0.01750083, -0.014178229, 0.03174632, 0.0935...",Enrollment,Site ID
1,8eeef31a-97fa-4fef-bcb7-a4b1c72b1369,4a5552de-4909-4846-bd25-3c28f9351b53,MVAL_DM018,Demographics,DM,Age Unit,AGEU,(DM.AGEU is enterable and missing),Field must not be missing when enterable,prompt user with ACTION DETAILS,<query the field for missing data>,Standard,/Standard/Copy of Otsuka Standard Edit Check S...,"[0.00561428, 0.014669912, 0.0441228, 0.056164,...","[-0.022254938, 0.017848019, 0.031054085, 0.029...",Enrollment,Participant ID
2,8eeef31a-97fa-4fef-bcb7-a4b1c72b1369,4a5552de-4909-4846-bd25-3c28f9351b53,MVAL_DM018,Demographics,DM,Age Unit,AGEU,(DM.AGEU is enterable and missing),Field must not be missing when enterable,prompt user with ACTION DETAILS,<query the field for missing data>,Standard,/Standard/Copy of Otsuka Standard Edit Check S...,"[0.00561428, 0.014669912, 0.0441228, 0.056164,...","[-0.022254938, 0.017848019, 0.031054085, 0.029...",Enrollment,Participant Number (Derived)
3,5322d11a-79e3-49a7-838c-0469661a9d4d,3c36705a-c794-48a0-aa54-0453ea51bde5,MVAL_SV010,Subject Visits,SV,Visit Date,VISDAT,(SV.VISDAT is an invalid date),Field must not be an invalid date such as 31Fe...,prompt user with ACTION DETAILS,<query the field for invalid date>,Standard,/Standard/Copy of Otsuka Standard Edit Check S...,"[-0.027394814416766167, 0.018084557726979256, ...","[-0.020077953, 0.0069108373, -0.023404991, -0....",Date of Visit,Visit date
4,fda81295-a0ab-4279-8947-ac699d5ae110,c3196559-2e4e-4c3e-bd98-afcd706447a2,MVAL_DS_IC006,Informed Consent,DS_IC,Was informed consent obtained?,IFCOCCUR,(DS_IC.IFCOCCUR is enterable and missing),Field must not be missing when enterable,prompt user with ACTION DETAILS,<query the field for missing data>,Standard,/Standard/Copy of Otsuka Standard Edit Check S...,"[0.02261987514793873, 0.021745149046182632, -0...","[0.002886054, 0.013917152, -0.0345182, 0.07630...",Informed Consent,Informed consent obtained?
5,fda81295-a0ab-4279-8947-ac699d5ae110,c3196559-2e4e-4c3e-bd98-afcd706447a2,MVAL_DS_IC006,Informed Consent,DS_IC,Was informed consent obtained?,IFCOCCUR,(DS_IC.IFCOCCUR is enterable and missing),Field must not be missing when enterable,prompt user with ACTION DETAILS,<query the field for missing data>,Standard,/Standard/Copy of Otsuka Standard Edit Check S...,"[0.02261987514793873, 0.021745149046182632, -0...","[0.002886054, 0.013917152, -0.0345182, 0.07630...",Informed Consent,Informed consent date
6,fda81295-a0ab-4279-8947-ac699d5ae110,c3196559-2e4e-4c3e-bd98-afcd706447a2,MVAL_DS_IC006,Informed Consent,DS_IC,Was informed consent obtained?,IFCOCCUR,(DS_IC.IFCOCCUR is enterable and missing),Field must not be missing when enterable,prompt user with ACTION DETAILS,<query the field for missing data>,Standard,/Standard/Copy of Otsuka Standard Edit Check S...,"[0.02261987514793873, 0.021745149046182632, -0...","[0.002886054, 0.013917152, -0.0345182, 0.07630...",Informed Consent,Informed consent time
7,62ea8058-50d8-4eb0-9b1b-275ae89847b3,bd0860d8-e7ce-4589-b23b-cbcd5931650b,MVAL_MH023,Medical History,MH,End Date,MHENDAT,(MH.MHENDAT is enterable and missing),Field must not be missing when enterable,prompt user with ACTION DETAILS,<query the field for missing data>,Standard,/Standard/Copy of Otsuka Standard Edit Check S...,"[0.021205350756645203, -0.014403634704649448, ...","[0.023139598, 0.020193553, 0.06760691, -0.0191...",Inform

In [239]:
import time
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed

start = time.time()
client = opensearch_client.opensearch_client
output_list = []
index_name = proj.get_variables()['local'].get('ecs_opensearch')
dataiku_project_var = "${projectKey}"

if dataiku_project_var in index_name:
    index_name = index_name.replace(
        dataiku_project_var,
        opensearch_client.project.project_key
    ).lower()

# Worker function for each row
def process_row(row):
    try:
        form_name = row['form_name']
        field_name = row['field_name']

        # Create embeddings
        form_emb = opensearch_client.create_embedding(
            form_name,
            proj.get_variables()['local'].get("default_embeddings_model_id")
        )
        field_emb = opensearch_client.create_embedding(
            field_name,
            proj.get_variables()['local'].get("default_embeddings_model_id")
        )

        form_vec = form_emb['response']
        field_vec = field_emb['response']

        # Build query
        query = {
            "query": {
                "bool": {
                    "should": [
                        {
                            "knn": {
                                "form_name_vector": {
                                    "vector": form_vec,
                                    "k": 10
                                }
                            }
                        },
                        {
                            "knn": {
                                "form_field_value_vector": {
                                    "vector": field_vec,
                                    "k": 10
                                }
                            }
                        }
                    ]
                }
            }
        }

        # Run search
        result = client.search(index=index_name, body=query, size=1)
        for hit in result["hits"]["hits"]:
            hit['_source']['original_form_name'] = form_name
            hit['_source']['original_field'] = field_name
            hit['_source']['score'] = hit['_score']
            return hit['_source']

    except Exception as e:
        print(f"Error processing row: {e}")
        return None

# Run in parallel
with ThreadPoolExecutor(max_workers=8) as executor:  # adjust workers
    futures = [executor.submit(process_row, row) for _, row in sub_data.iterrows()]
    for future in as_completed(futures):
        res = future.result()
        if res:
            output_list.append(res)

# Convert to DataFrame
final_df = pd.DataFrame(output_list)

end = time.time()
print(f"Time taken: {end - start:.2f} seconds")


/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made t

/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made t

/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made t

/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made t

/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made t

Time taken: 2.66 seconds


/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made t

In [240]:
final_df

,ecs_id,form_id,validation_id,form_name,form_domain_name,form_field_value,variable_name,validation_logic,reasoning,action,action_details,source,path,form_name_vector,form_field_value_vector,original_form_name,original_field,score
0,8eeef31a-97fa-4fef-bcb7-a4b1c72b1369,4a5552de-4909-4846-bd25-3c28f9351b53,MVAL_DM018,Demographics,DM,Age Unit,AGEU,(DM.AGEU is enterable and missing),Field must not be missing when enterable,prompt user with ACTION DETAILS,<query the field for missing data>,Standard,/Standard/Copy of Otsuka Standard Edit Check S...,"[0.00561428, 0.014669912, 0.0441228, 0.056164,...","[-0.022254938, 0.017848019, 0.031054085, 0.029...",Enrollment,Participant ID,1.247537
1,fda81295-a0ab-4279-8947-ac699d5ae110,c3196559-2e4e-4c3e-bd98-afcd706447a2,MVAL_DS_IC006,Informed Consent,DS_IC,Was informed consent obtained?,IFCOCCUR,(DS_IC.IFCOCCUR is enterable and missing),Field must not be missing when enterable,prompt user with ACTION DETAILS,<query the field for missing data>,Standard,/Standard/Copy of Otsuka Standard Edit Check S...,"[0.02261987514793873, 0.021745149046182632, -0...","[0.002886054, 0.013917152, -0.0345182, 0.07630...",Informed Consent,Informed consent obtained?,1.847901
2,fda81295-a0ab-4279-8947-ac699d5ae110,c3196559-2e4e-4c3e-bd98-afcd706447a2,MVAL_DS_IC006,Informed Consent,DS_IC,Was informed consent obtained?,IFCOCCUR,(DS_IC.IFCOCCUR is enterable and missing),Field must not be missing when enterable,prompt user with ACTION DETAILS,<query the field for missing data>,Standard,/Standard/Copy of Otsuka Standard Edit Check S...,"[0.02261987514793873, 0.021745149046182632, -0...","[0.002886054, 0.013917152, -0.0345182, 0.07630...",Informed Consent,Informed consent date,1.693060
3,fda81295-a0ab-4279-8947-ac699d5ae110,c3196559-2e4e-4c3e-bd98-afcd706447a2,MVAL_DS_IC006,Informed Consent,DS_IC,Was informed consent obtained?,IFCOCCUR,(DS_IC.IFCOCCUR is enterable and missing),Field must not be missing when enterable,prompt user with ACTION DETAILS,<query the field for missing data>,Standard,/Standard/Copy of Otsuka Standard Edit Check S...,"[0.02261987514793873, 0.021745149046182632, -0...","[0.002886054, 0.013917152, -0.0345182, 0.07630...",Informed Consent,Informed consent time,1.666454
4,8eeef31a-97fa-4fef-bcb7-a4b1c72b1369,4a5552de-4909-4846-bd25-3c28f9351b53,MVAL_DM018,Demographics,DM,Age Unit,AGEU,(DM.AGEU is enterable and missing),Field must not be missing when enterable,prompt user with ACTION DETAILS,<query the field for missing data>,Standard,/Standard/Copy of Otsuka Standard Edit Check S...,"[0.00561428, 0.014669912, 0.0441228, 0.056164,...","[-0.022254938, 0.017848019, 0.031054085, 0.029...",Enrollment,Participant Number (Derived),1.244726
5,fda81295-a0ab-4279-8947-ac699d5ae110,c3196559-2e4e-4c3e-bd98-afcd706447a2,MVAL_DS_IC006,Informed Consent,DS_IC,Was informed consent obtained?,IFCOCCUR,(DS_IC.IFCOCCUR is enterable and missing),Field must not be missing when enterable,prompt user with ACTION DETAILS,<query the field for missing data>,Standard,/Standard/Copy of Otsuka Standard Edit Check S...,"[0.02261987514793873, 0.021745149046182632, -0...","[0.002886054, 0.013917152, -0.0345182, 0.07630...",Informed Consent,Informed consent version number,1.661254
6,44c40b95-e715-4892-bcd4-6400447a965e,4a5552de-4909-4846-bd25-3c28f9351b53,MVAL_DM027,Demographics,DM,Country,COUNTRY,(DM.COUNTRY is enterable and missing),Field must not be missing when enterable,prompt user with ACTION DETAILS,<query the field for missing data>,Standard,/Standard/Copy of Otsuka Standard Edit Check S...,"[0.00561428, 0.014669912, 0.0441228, 0.056164,...","[-0.01750083, -0.014178229, 0.03174632, 0.0935...",Enrollment,Site ID,1.199591
7,62ea8058-50d8-4eb0-9b1b-275ae89847b3,bd0860d8-e7ce-4589-b23b-cbcd5931650b,MVAL_MH023,Medical History,MH,End Date,MHENDAT,(MH.MHENDAT is enterable and missing),Field must not be missing when enterable,prompt user with ACTION DETAILS,<query the field for missing data>,Standard,/Standard/Copy of Otsuka Standard E

In [243]:
fil_res_df = final_df[["original_form_name", "original_field","form_name","form_field_value",'score']]
fil_res_df['score'] = fil_res_df['score']/2
fil_res_df

/tmp/ipykernel_3513200/3986629152.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fil_res_df['score'] = fil_res_df['score']/2


,original_form_name,original_field,form_name,form_field_value,score
0,Enrollment,Participant ID,Demographics,Age Unit,0.623769
1,Informed Consent,Informed consent obtained?,Informed Consent,Was informed consent obtained?,0.923950
2,Informed Consent,Informed consent date,Informed Consent,Was informed consent obtained?,0.846530
3,Informed Consent,Informed consent time,Informed Consent,Was informed consent obtained?,0.833227
4,Enrollment,Participant Number (Derived),Demographics,Age Unit,0.622363
5,Informed Consent,Informed consent version number,Informed Consent,Was informed consent obtained?,0.830627
6,Enrollment,Site ID,Demographics,Country,0.599796
7,Informed Consent,Derived date,Medical History,End Date,0.651459
8,Date of Visit,Visit date,Subject Visits,Visit Date,0.792303
9,Informed Consent,Protocol version,Informed Consent,Protocol Version Number,0.905990


In [231]:
# Clean both DataFrame column and your list
data["form_name_clean"] = data["form_name"].str.strip().str.lower()

forms = [
    "Subject Visits",
    "Informed Consent",
    "Demographics",
    "Screen Failure",
    "Medical History",
    "Adverse Event",
    "Prior and Concomitant Medications",
    "Post-treatment Follow-up"
]

forms_clean = [f.strip().lower() for f in forms]

# Now filter
filtered_df = data[data["form_name_clean"].isin(forms_clean)]


In [228]:
filtered_df = data[data["form_name"].str.lower().isin([f.lower() for f in forms])]


In [232]:
filtered_df

,form_name,field_name,form_name_clean
4,Informed Consent,Informed consent obtained?,informed consent
5,Informed Consent,Informed consent date,informed consent
6,Informed Consent,Informed consent time,informed consent
7,Informed Consent,Derived date,informed consent
8,Informed Consent,Informed consent version number,informed consent
9,Informed Consent,Standardized disposition term,informed consent
10,Informed Consent,Protocol version,informed consent
11,Demographics,Date of Birth,demographics
12,Demographics,Age,demographics
13,Demographics,Sex at Birth,demographics


In [233]:
print(data["form_name"].unique())


['Enrollment ' 'Date of Visit ' 'Informed Consent ' 'Demographics '
 'Inclusion/Exclusion Criteria ' 'Medical and Surgical History '
 'Physical Examination ' 'Vital Signs ' 'Weight/Height/BMI '
 'Electrocardiogram ' 'Laboratory Test Collections Screening '
 'Laboratory Test_DOA ' 'Laboratory Test_PG '
 'Laboratory Test Collection_FSH ' 'Screening Outcome '
 'Columbia Suicide Severity Rating Scale (C-SSRS)_Baseline/Screening '
 'Weight ' 'Triplicate Electrocardiogram (DM1) '
 'Laboratory Test Collections ' 'Screening DM1 Outcome '
 'Columbia Suicide Severity Rating Scale (C-SSRS) _Since Last Visit '
 'Treatment Arm ' 'Randomization (Arm 1, 2 and 5) '
 'Randomization (Arm 3 and 4) ' 'Meal_Arm 2 ' 'Dosing (QD)_Arm 2 '
 'Dosing (QD) ' 'Dosing (BID)_(Arm 3 and Arm 5) ' 'Vital Signs (D1) '
 'Pharmacokinetics (D1) ' 'Pharmacokinetics (D2) '
 'Pharmacokinetics (D3) ' 'Pharmacokinetics (D3)_Arm 5 '
 'Vital Signs (D4) ' 'Pharmacokinetics (D4) '
 'Pharmacokinetics (D4)_Arm 5 ' 'Pharmacokinetics (

In [234]:
import time
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed

start = time.time()
client = opensearch_client.opensearch_client
output_list = []
index_name = proj.get_variables()['local'].get('ecs_opensearch')
dataiku_project_var = "${projectKey}"

if dataiku_project_var in index_name:
    index_name = index_name.replace(
        dataiku_project_var,
        opensearch_client.project.project_key
    ).lower()

# Worker function for each row
def process_row(row):
    try:
        form_name = row['form_name']
        field_name = row['field_name']

        # Create embeddings
        form_emb = opensearch_client.create_embedding(
            form_name,
            proj.get_variables()['local'].get("default_embeddings_model_id")
        )
        field_emb = opensearch_client.create_embedding(
            field_name,
            proj.get_variables()['local'].get("default_embeddings_model_id")
        )

        form_vec = form_emb['response']
        field_vec = field_emb['response']

        # Build query
        query = {
            "query": {
                "bool": {
                    "should": [
                        {
                            "knn": {
                                "form_name_vector": {
                                    "vector": form_vec,
                                    "k": 10
                                }
                            }
                        },
                        {
                            "knn": {
                                "form_field_value_vector": {
                                    "vector": field_vec,
                                    "k": 10
                                }
                            }
                        }
                    ]
                }
            }
        }

        # Run search
        result = client.search(index=index_name, body=query, size=1)
        for hit in result["hits"]["hits"]:
            hit['_source']['original_form_name'] = form_name
            hit['_source']['original_field'] = field_name
            return hit['_source']

    except Exception as e:
        print(f"Error processing row: {e}")
        return None

# Run in parallel
with ThreadPoolExecutor(max_workers=4) as executor:  # adjust workers
    futures = [executor.submit(process_row, row) for _, row in filtered_df.iterrows()]
    for future in as_completed(futures):
        res = future.result()
        if res:
            output_list.append(res)

# Convert to DataFrame
filter_final_df_ = pd.DataFrame(output_list)

end = time.time()
print(f"Time taken: {end - start:.2f} seconds")


/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made t

Time taken: 0.92 seconds


/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made t

In [235]:
filter_final_df_

,ecs_id,form_id,validation_id,form_name,form_domain_name,form_field_value,variable_name,validation_logic,reasoning,action,action_details,source,path,form_name_vector,form_field_value_vector,original_form_name,original_field
0,fda81295-a0ab-4279-8947-ac699d5ae110,c3196559-2e4e-4c3e-bd98-afcd706447a2,MVAL_DS_IC006,Informed Consent,DS_IC,Was informed consent obtained?,IFCOCCUR,(DS_IC.IFCOCCUR is enterable and missing),Field must not be missing when enterable,prompt user with ACTION DETAILS,<query the field for missing data>,Standard,/Standard/Copy of Otsuka Standard Edit Check S...,"[0.02261987514793873, 0.021745149046182632, -0...","[0.002886054, 0.013917152, -0.0345182, 0.07630...",Informed Consent,Informed consent time
1,fda81295-a0ab-4279-8947-ac699d5ae110,c3196559-2e4e-4c3e-bd98-afcd706447a2,MVAL_DS_IC006,Informed Consent,DS_IC,Was informed consent obtained?,IFCOCCUR,(DS_IC.IFCOCCUR is enterable and missing),Field must not be missing when enterable,prompt user with ACTION DETAILS,<query the field for missing data>,Standard,/Standard/Copy of Otsuka Standard Edit Check S...,"[0.02261987514793873, 0.021745149046182632, -0...","[0.002886054, 0.013917152, -0.0345182, 0.07630...",Informed Consent,Informed consent obtained?
2,62ea8058-50d8-4eb0-9b1b-275ae89847b3,bd0860d8-e7ce-4589-b23b-cbcd5931650b,MVAL_MH023,Medical History,MH,End Date,MHENDAT,(MH.MHENDAT is enterable and missing),Field must not be missing when enterable,prompt user with ACTION DETAILS,<query the field for missing data>,Standard,/Standard/Copy of Otsuka Standard Edit Check S...,"[0.021205350756645203, -0.014403634704649448, ...","[0.023139598, 0.020193553, 0.06760691, -0.0191...",Informed Consent,Derived date
3,fda81295-a0ab-4279-8947-ac699d5ae110,c3196559-2e4e-4c3e-bd98-afcd706447a2,MVAL_DS_IC006,Informed Consent,DS_IC,Was informed consent obtained?,IFCOCCUR,(DS_IC.IFCOCCUR is enterable and missing),Field must not be missing when enterable,prompt user with ACTION DETAILS,<query the field for missing data>,Standard,/Standard/Copy of Otsuka Standard Edit Check S...,"[0.02261987514793873, 0.021745149046182632, -0...","[0.002886054, 0.013917152, -0.0345182, 0.07630...",Informed Consent,Informed consent date
4,fda81295-a0ab-4279-8947-ac699d5ae110,c3196559-2e4e-4c3e-bd98-afcd706447a2,MVAL_DS_IC006,Informed Consent,DS_IC,Was informed consent obtained?,IFCOCCUR,(DS_IC.IFCOCCUR is enterable and missing),Field must not be missing when enterable,prompt user with ACTION DETAILS,<query the field for missing data>,Standard,/Standard/Copy of Otsuka Standard Edit Check S...,"[0.02261987514793873, 0.021745149046182632, -0...","[0.002886054, 0.013917152, -0.0345182, 0.07630...",Informed Consent,Informed consent version number
5,22df2dda-056c-48cc-88d6-029fbfa872f2,c3196559-2e4e-4c3e-bd98-afcd706447a2,MVAL_DS_IC004,Informed Consent,DS_IC,Type of Consent,DSSCAT,(DS_IC.DSSCAT is enterable and missing),Field must not be missing when enterable,prompt user with ACTION DETAILS,<query the field for missing data>,Standard,/Standard/Copy of Otsuka Standard Edit Check S...,"[0.02261987514793873, 0.021745149046182632, -0...","[0.028999867, 0.044233263, 0.0076308018, 0.063...",Informed Consent,Standardized disposition term
6,2a2f0324-6fb3-406e-85cd-ea8da52b82c5,4a5552de-4909-4846-bd25-3c28f9351b53,MVAL_DM007,Demographics,DM,Birth Date,BRTHDAT,(DM.BRTHDAT is a future date),Field must not be a date in the future,prompt user with ACTION DETAILS,<query the field for future date>,Standard,/Standard/Copy of Otsuka Standard Edit Check S...,"[0.00561428, 0.014669912, 0.0441228, 0.056164,...","[0.02373592, -0.0060537606, 0.037967116, 0.023...",Demographics,Date of Birth
7,426a6d30-e7db-407b-b10e-9c89ca760ce3,c3196559-2e4e-4c3e-bd98-afcd706447a2,MVAL_DS_IC005,Informed Consent,DS_IC,Protocol Version Number,QVAL_PROTVER,(DS_IC.QVAL_PROTVER is enterable and missing),Field must not be missing when enterable,prompt user with ACTION DETAILS,<query the field for missing data>,Standard,/Standard/Copy of Otsuka 

In [236]:
fil_res_df = filter_final_df_[["original_form_name", "original_field","form_name","form_field_value"]]
fil_res_df

,original_form_name,original_field,form_name,form_field_value
0,Informed Consent,Informed consent time,Informed Consent,Was informed consent obtained?
1,Informed Consent,Informed consent obtained?,Informed Consent,Was informed consent obtained?
2,Informed Consent,Derived date,Medical History,End Date
3,Informed Consent,Informed consent date,Informed Consent,Was informed consent obtained?
4,Informed Consent,Informed consent version number,Informed Consent,Was informed consent obtained?
5,Informed Consent,Standardized disposition term,Informed Consent,Type of Consent
6,Demographics,Date of Birth,Demographics,Birth Date
7,Informed Consent,Protocol version,Informed Consent,Protocol Version Number
8,Demographics,Age,Demographics,Age
9,Demographics,Sex at Birth,Demographics,Sex at Birth
